In [1]:
import pandas as pd
import numpy as np
df = pd.read_excel('/content/cases_train2 (5).xlsx')
df['Оценка'] = df['Оценка'].round().astype('Int64')
df_test_cases = pd.read_excel('/content/Test1_2 (3).xlsx')
df_test_cases['Оценка'] = df_test_cases['Оценка'].round().astype('Int64')
#df = pd.read_excel("/content/good table.xlsx")
df.head(100)


,Кейс,Решение,Решение кейса,ЦА,Проработка решения,Финансовая модель и метрики,Анализ рисков,Доказательства,Оценка
0,1,1,Я выбрал отрасль туризма и гостиничного бизнес...,1,1,1,1,1,1
1,1,2,Я выбрал для разработки отраслевого решения сф...,2,2,2,2,2,2
2,1,3,"Я выбрал отрасль образования, а именно сегмент...",2,4,4,3,3,3
3,1,4,Я выбрал отрасль розничной торговли продуктами...,1,3,2,2,1,2
4,1,5,Я выбрал для отраслевого решения сферу гостини...,3,2,2,2,3,2
...,...,...,...,...,...,...,...,...,...
95,1,96,Я выбрал отрасль грузоперевозок. Целевая аудит...,2,2,4,3,1,2
96,1,97,Я выбрал отрасль ремонта техники. Целевая ауди...,3,2,3,2,2,2
97,1,98,Я выбрал отрасль такси. Целевая аудитория: вод...,1,1,4,3,1,2
98,1,99,Я выбрал отрасль ремонта квартир. Целевая ауди...,3,3,3,2,2,3


In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [3]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [ ]:
df["Оценка"].value_counts()

,count
Оценка,
4,208
3,206
1,110
2,105
5,94
7,26
6,21
8,21
9,18


In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


In [4]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [5]:
class BertClassifier(nn.Module):
  def __init__(self, num_classes, model_name='bert-base-multilingual-cased'):
    super(BertClassifier, self).__init__()
    self.bert = BertModel.from_pretrained(model_name)
    for param in self.bert.parameters():
      param.requires_grad=False
    self.classifier = nn.Sequential(nn.Dropout(0.3),
                                    nn.Linear(self.bert.config.hidden_size, 256),
                                    nn.ReLU(),
                                    nn.Dropout(0.2),
                                    nn.Linear(256, num_classes)
    )
  def forward(self, input_ids, attention_mask):
    with torch.no_grad():
      outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
      cls = outputs.last_hidden_state[:, 0, :]
    logits = self.classifier(cls)
    return logits

In [6]:
tokenizer= BertTokenizer.from_pretrained("bert-base-multilingual-cased")
model = BertClassifier(num_classes=5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def train(loader):
    total_loss = 0
    model.train()
    for batch in loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)
      optimizer.zero_grad()
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    return total_loss / len(loader)


In [8]:
def evals(loader):
    model.eval()
    loss_lst = []
    predictions = []
    total = 0
    correct = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            preds = torch.argmax(outputs, dim=1)

            loss_lst.append(loss.item())
            correct += (preds == labels).sum().item()
            total += len(labels)
            predictions.extend(preds.cpu().numpy())

    return np.mean(loss_lst), correct / total, np.array(predictions)

In [9]:
def predicts(texts, model, tokenizer, device, label_encoder=None, max_len=128):
  model.eval()
  predictions = []
  with torch.no_grad():
    for text in texts:
      encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
      input_ids = encoding["input_ids"].to(device)
      attention_mask = encoding["attention_mask"].to(device)
      outputs = model(input_ids=input_ids, attention_mask=attention_mask)
      probs = torch.softmax(outputs, dim=1)
      pred = torch.argmax(probs, dim=1)
      predictions.append(pred.cpu().numpy()[0])
  return np.array(predictions)

In [10]:
X_text_audience = df['Решение кейса'].fillna('').values
y_audience = (df['ЦА'].values - 1).astype("int64")

X_train_text_audience, X_test_text_audience, y_train_audience, y_test_audience = train_test_split(
    X_text_audience, y_audience, test_size=0.2, random_state=42
)
X_text_dataset_audience = TextDataset(X_train_text_audience, y_train_audience, tokenizer)
X_val_dataset_audience = TextDataset(X_test_text_audience, y_test_audience, tokenizer)
X_text_dataloader_audience = DataLoader(X_text_dataset_audience, batch_size=16, shuffle=True)
X_val_dataloader_audience = DataLoader(X_val_dataset_audience, batch_size=16, shuffle=False)
model_audience = BertClassifier(num_classes=5)
model_audience.to(device)
optimizer_audience = torch.optim.AdamW(model_audience.classifier.parameters(), lr=2e-4)
criterion_audience = nn.CrossEntropyLoss()
model = model_audience
model.to(device)
optimizer = optimizer_audience
criterion = criterion_audience


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import pandas as pd
import numpy as np
from collections import Counter

def check_data_integrity(X_text, y, tokenizer, max_len=128):
    """
    Полная проверка данных перед обучением
    """
    print("="*60)
    print("🔍 ПРОВЕРКА ЦЕЛОСТНОСТИ ДАННЫХ")
    print("="*60)

    # ============ 1. РАЗМЕРЫ ============
    print("\n1️⃣ РАЗМЕРЫ ДАННЫХ:")
    print(f"   Количество текстов: {len(X_text)}")
    print(f"   Количество меток: {len(y)}")

    if len(X_text) != len(y):
        print(f"   ❌ ОШИБКА: X и y имеют разный размер!")
        return False
    else:
        print(f"   ✅ X и y совпадают по размеру")

    # ============ 2. ПУСТЫЕ ЗНАЧЕНИЯ ============
    print("\n2️⃣ ПРОВЕРКА НА ПУСТЫЕ ЗНАЧЕНИЯ:")

    # Пустые тексты
    empty_texts = [i for i, t in enumerate(X_text) if pd.isna(t) or str(t).strip() == '']
    print(f"   Пустых текстов: {len(empty_texts)}")
    if len(empty_texts) > 0:
        print(f"   ⚠️ Индексы пустых текстов: {empty_texts[:10]}")
    else:
        print(f"   ✅ Нет пустых текстов")

    # Пустые метки
    empty_labels = [i for i, l in enumerate(y) if pd.isna(l)]
    print(f"   Пустых меток: {len(empty_labels)}")
    if len(empty_labels) > 0:
        print(f"   ❌ Есть пустые метки!")
    else:
        print(f"   ✅ Нет пустых меток")

    # ============ 3. МЕТКИ ============
    print("\n3️⃣ ПРОВЕРКА МЕТОК:")

    unique_labels = np.unique(y)
    print(f"   Уникальные метки: {unique_labels}")
    print(f"   Количество классов: {len(unique_labels)}")
    print(f"   Минимальная метка: {unique_labels.min()}")
    print(f"   Максимальная метка: {unique_labels.max()}")

    # Проверка, что метки начинаются с 0
    if unique_labels.min() != 0:
        print(f"   ⚠️ Метки начинаются с {unique_labels.min()}, а должны с 0")
        print(f"   📌 Исправьте: y = y - {unique_labels.min()}")

    # Проверка на пропуски в метках
    expected_labels = set(range(len(unique_labels)))
    actual_labels = set(unique_labels)
    if expected_labels != actual_labels:
        missing = expected_labels - actual_labels
        extra = actual_labels - expected_labels
        if missing:
            print(f"   ⚠️ Отсутствуют метки: {missing}")
        if extra:
            print(f"   ⚠️ Лишние метки: {extra}")
    else:
        print(f"   ✅ Все метки идут подряд от 0 до {len(unique_labels)-1}")

    # Распределение меток
    label_counts = Counter(y)
    print(f"\n   📊 Распределение меток:")
    for label in sorted(label_counts.keys()):
        count = label_counts[label]
        percent = count / len(y) * 100
        bar = "█" * int(percent / 2)
        print(f"      Класс {label}: {count:5d} ({percent:5.1f}%) {bar}")

    # ============ 4. ТЕКСТЫ ============
    print("\n4️⃣ ПРОВЕРКА ТЕКСТОВ:")

    # Длины текстов
    text_lengths = [len(str(t).split()) for t in X_text]
    print(f"   Минимальная длина: {min(text_lengths)} слов")
    print(f"   Максимальная длина: {max(text_lengths)} слов")
    print(f"   Средняя длина: {np.mean(text_lengths):.1f} слов")
    print(f"   Медианная длина: {np.median(text_lengths):.1f} слов")

    # Тексты длиннее max_len
    long_texts = sum([1 for t in text_lengths if t > max_len])
    print(f"   Текстов длиннее {max_len} токенов: {long_texts} ({long_texts/len(X_text)*100:.1f}%)")

    # Проверка на дубликаты
    unique_texts = len(set([str(t) for t in X_text]))
    print(f"   Уникальных текстов: {unique_texts} из {len(X_text)}")
    if unique_texts < len(X_text):
        print(f"   ⚠️ Есть {len(X_text) - unique_texts} дубликатов!")

    # ============ 5. ТОКЕНИЗАЦИЯ ============
    print("\n5️⃣ ПРОВЕРКА ТОКЕНИЗАЦИИ:")

    # Тест токенизации на первом тексте
    sample_text = str(X_text[0]) if len(X_text) > 0 else ""
    if sample_text:
        encoding = tokenizer(
            sample_text,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )
        print(f"   Пример текста: {sample_text[:100]}...")
        print(f"   Длина токенов: {encoding['input_ids'].shape[1]}")
        print(f"   Пример токенов: {encoding['input_ids'][0][:10]}")
        print(f"   ✅ Токенизация работает")

    # ============ 6. СООТВЕТСТВИЕ ТИПОВ ============
    print("\n6️⃣ ПРОВЕРКА ТИПОВ ДАННЫХ:")
    print(f"   Тип X: {type(X_text)}")
    print(f"   Тип y: {type(y)}")
    print(f"   Тип первого текста: {type(X_text[0]) if len(X_text) > 0 else 'None'}")
    print(f"   Тип первой метки: {type(y[0]) if len(y) > 0 else 'None'}")
    print(f"   Тип y: {y.dtype if hasattr(y, 'dtype') else 'list'}")

    # ============ 7. ПРОВЕРКА ДИСБАЛАНСА ============
    print("\n7️⃣ ПРОВЕРКА ДИСБАЛАНСА КЛАССОВ:")
    min_count = min(label_counts.values())
    max_count = max(label_counts.values())
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

    print(f"   Минимальное количество: {min_count}")
    print(f"   Максимальное количество: {max_count}")
    print(f"   Соотношение max/min: {imbalance_ratio:.2f}")

    if imbalance_ratio > 3:
        print(f"   ⚠️ Сильный дисбаланс классов! (>3)")
        print(f"   📌 Рекомендуется использовать stratify при разбиении")
    else:
        print(f"   ✅ Дисбаланс приемлемый")

    # ============ 8. ИТОГ ============
    print("\n" + "="*60)
    print("📊 ИТОГОВЫЙ ВЕРДИКТ:")

    all_checks_passed = (
        len(X_text) == len(y) and
        len(empty_texts) == 0 and
        len(empty_labels) == 0 and
        unique_labels.min() == 0 and
        len(unique_labels) == max(unique_labels) + 1
    )

    if all_checks_passed:
        print("✅ ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! Данные готовы к обучению.")
    else:
        print("⚠️ ЕСТЬ ПРОБЛЕМЫ, которые нужно исправить:")
        if len(X_text) != len(y):
            print("   - Размер X и y не совпадают")
        if len(empty_texts) > 0:
            print(f"   - Есть {len(empty_texts)} пустых текстов")
        if len(empty_labels) > 0:
            print(f"   - Есть {len(empty_labels)} пустых меток")
        if unique_labels.min() != 0:
            print(f"   - Метки начинаются с {unique_labels.min()}, а должны с 0")

    print("="*60)
    return all_checks_passed

# ============= ЗАПУСК ПРОВЕРКИ =============
check_data_integrity(X_text_audience, y_audience, tokenizer, max_len=128)

🔍 ПРОВЕРКА ЦЕЛОСТНОСТИ ДАННЫХ

1️⃣ РАЗМЕРЫ ДАННЫХ:
   Количество текстов: 500
   Количество меток: 500
   ✅ X и y совпадают по размеру

2️⃣ ПРОВЕРКА НА ПУСТЫЕ ЗНАЧЕНИЯ:
   Пустых текстов: 0
   ✅ Нет пустых текстов
   Пустых меток: 0
   ✅ Нет пустых меток

3️⃣ ПРОВЕРКА МЕТОК:
   Уникальные метки: [0 1 2 3 4]
   Количество классов: 5
   Минимальная метка: 0
   Максимальная метка: 4
   ✅ Все метки идут подряд от 0 до 4

   📊 Распределение меток:
      Класс 0:    64 ( 12.8%) ██████
      Класс 1:    63 ( 12.6%) ██████
      Класс 2:   127 ( 25.4%) ████████████
      Класс 3:   140 ( 28.0%) ██████████████
      Класс 4:   106 ( 21.2%) ██████████

4️⃣ ПРОВЕРКА ТЕКСТОВ:
   Минимальная длина: 15 слов
   Максимальная длина: 1854 слов
   Средняя длина: 139.6 слов
   Медианная длина: 81.0 слов
   Текстов длиннее 128 токенов: 148 (29.6%)
   Уникальных текстов: 477 из 500
   ⚠️ Есть 23 дубликатов!

5️⃣ ПРОВЕРКА ТОКЕНИЗАЦИИ:
   Пример текста: Я выбрал отрасль туризма и гостиничного бизнеса. Целевая

np.True_

In [ ]:
# Проверка устройства
print(f"Модель на устройстве: {next(model.parameters()).device}")
print(f"Данные на устройстве: {batch['input_ids'].device}")
print(f"Устройство: {device}")

# Убедитесь, что все на одном устройстве
model = model.to(device)

Модель на устройстве: cuda:0


NameError: name 'batch' is not defined

In [11]:
best_loss_audience = float("inf")
for epoch in range(10):
  train_loss = train(X_text_dataloader_audience)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_audience)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_audience:
    best_loss_audience = val_loss
    torch.save(model_audience.state_dict(), "best_model_audience.pt")


train_loss= 1.5878752279281616
val_loss= 1.5752601623535156
val_acc= 0.22
train_loss= 1.5539688920974732
val_loss= 1.576516934803554
val_acc= 0.22
train_loss= 1.5293539190292358
val_loss= 1.553766403879438
val_acc= 0.22
train_loss= 1.5299545049667358
val_loss= 1.546446783202035
val_acc= 0.22
train_loss= 1.5101185512542725
val_loss= 1.5377516065325056
val_acc= 0.23
train_loss= 1.4865691709518432
val_loss= 1.5152010747364588
val_acc= 0.28
train_loss= 1.480666871070862
val_loss= 1.4971037251608712
val_acc= 0.28
train_loss= 1.4619708633422852
val_loss= 1.4949731826782227
val_acc= 0.28
train_loss= 1.461350440979004
val_loss= 1.4749408108847482
val_acc= 0.39
train_loss= 1.430920009613037
val_loss= 1.4801606621061052
val_acc= 0.28


In [12]:
model_audience.load_state_dict(torch.load("best_model_audience.pt"))
model_audience.to(device)
model_audience.eval()
_, _, predictions = evals(X_val_dataloader_audience)
true_labels = []
for batch in X_val_dataloader_audience:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.75      0.27      0.40        11
           1       0.00      0.00      0.00        13
           2       0.67      0.06      0.11        32
           3       0.31      0.77      0.44        22
           4       0.45      0.77      0.57        22

    accuracy                           0.39       100
   macro avg       0.43      0.38      0.30       100
weighted avg       0.46      0.39      0.30       100

MAE 1.04


In [13]:
X_text_sol = df['Решение кейса'].fillna('').values
y_sol = (df['Проработка решения'].values - 1).astype("int64")

X_train_text_sol, X_test_text_sol, y_train_sol, y_test_sol = train_test_split(
    X_text_sol, y_sol, test_size=0.2, random_state=42
)
X_text_dataset_sol = TextDataset(X_train_text_sol, y_train_sol, tokenizer)
X_val_dataset_sol = TextDataset(X_test_text_sol, y_test_sol, tokenizer)
X_text_dataloader_sol = DataLoader(X_text_dataset_sol, batch_size=16, shuffle=True)
X_val_dataloader_sol = DataLoader(X_val_dataset_sol, batch_size=16, shuffle=False)

In [14]:
model_sol = BertClassifier(num_classes=5)
model_sol.to(device)
optimizer_sol = torch.optim.AdamW(model_sol.classifier.parameters(), lr=2e-4)
criterion_sol = nn.CrossEntropyLoss()
model = model_sol
model.to(device)
optimizer = optimizer_sol
criterion = criterion_sol


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
best_loss_sol = float("inf")
for epoch in range(10):
  train_loss = train(X_text_dataloader_sol)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_sol)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_sol:
    best_loss_sol = val_loss
    torch.save(model_sol.state_dict(), "best_model_sol.pt")


train_loss= 1.598867621421814
val_loss= 1.564971651349749
val_acc= 0.28
train_loss= 1.5623697710037232
val_loss= 1.5561882768358504
val_acc= 0.27
train_loss= 1.5530318737030029
val_loss= 1.5458035809653146
val_acc= 0.29
train_loss= 1.532660174369812
val_loss= 1.521946668624878
val_acc= 0.3
train_loss= 1.53346736907959
val_loss= 1.5166438136781966
val_acc= 0.3
train_loss= 1.5057503700256347
val_loss= 1.5127986328942435
val_acc= 0.32
train_loss= 1.4803126287460326
val_loss= 1.4983681099755424
val_acc= 0.33
train_loss= 1.4668378114700318
val_loss= 1.4828108719417028
val_acc= 0.38
train_loss= 1.4652547550201416
val_loss= 1.4730465752737862
val_acc= 0.31
train_loss= 1.4476465606689453
val_loss= 1.4642682926995414
val_acc= 0.48


In [16]:
model_sol.load_state_dict(torch.load("best_model_sol.pt"))
model_sol.to(device)
model_sol.eval()
_, _, predictions = evals(X_val_dataloader_sol)
true_labels = []
for batch in X_val_dataloader_sol:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.50      0.42      0.45        12
           1       0.00      0.00      0.00        16
           2       0.53      0.72      0.61        29
           3       0.44      0.41      0.42        27
           4       0.44      0.69      0.54        16

    accuracy                           0.48       100
   macro avg       0.38      0.45      0.40       100
weighted avg       0.40      0.48      0.43       100

MAE 0.82


In [17]:
model_finance = BertClassifier(num_classes=5)
model_finance.to(device)
optimizer_finance = torch.optim.AdamW(model_finance.classifier.parameters(), lr=2e-4)
criterion_finance = nn.CrossEntropyLoss()
model = model_finance
model.to(device)
optimizer = optimizer_finance
criterion = criterion_finance


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
X_text_finance = df['Решение кейса'].fillna('').values
y_finance = (df['Финансовая модель и метрики'].values - 1).astype("int64")

X_train_text_finance, X_test_text_finance, y_train_finance, y_test_finance = train_test_split(
    X_text_finance, y_finance, test_size=0.2, random_state=42
)
X_text_dataset_finance = TextDataset(X_train_text_finance, y_train_finance, tokenizer)
X_val_dataset_finance = TextDataset(X_test_text_finance, y_test_finance, tokenizer)
X_text_dataloader_finance = DataLoader(X_text_dataset_finance, batch_size=16, shuffle=True)
X_val_dataloader_finance = DataLoader(X_val_dataset_finance, batch_size=16, shuffle=False)

In [19]:
best_loss_finance = float("inf")
for epoch in range(10):
  train_loss = train(X_text_dataloader_finance)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_finance)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_finance:
    best_loss_finance = val_loss
    torch.save(model_finance.state_dict(), "best_model_finance.pt")


train_loss= 1.5933920907974244
val_loss= 1.5142018795013428
val_acc= 0.31
train_loss= 1.5471098804473877
val_loss= 1.4982974358967371
val_acc= 0.31
train_loss= 1.5445300674438476
val_loss= 1.4852500983646937
val_acc= 0.33
train_loss= 1.5104664850234986
val_loss= 1.4739373070853097
val_acc= 0.39
train_loss= 1.499237937927246
val_loss= 1.460028852735247
val_acc= 0.42
train_loss= 1.493523063659668
val_loss= 1.4528080906186784
val_acc= 0.4
train_loss= 1.4826540374755859
val_loss= 1.4407944508961268
val_acc= 0.38
train_loss= 1.4562873458862304
val_loss= 1.4339187826429094
val_acc= 0.38
train_loss= 1.4544736337661743
val_loss= 1.4190654414040702
val_acc= 0.44
train_loss= 1.4499148035049438
val_loss= 1.4124186209269933
val_acc= 0.38


In [20]:
model_finance.load_state_dict(torch.load("best_model_finance.pt"))
model_finance.to(device)
model_finance.eval()
_, _, predictions = evals(X_val_dataloader_finance)
true_labels = []
for batch in X_val_dataloader_finance:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.67      0.29      0.40        14
           1       0.00      0.00      0.00        15
           2       0.35      0.88      0.50        33
           3       0.42      0.17      0.24        30
           4       0.00      0.00      0.00         8

    accuracy                           0.38       100
   macro avg       0.29      0.27      0.23       100
weighted avg       0.34      0.38      0.29       100

MAE 0.83


In [21]:
model_risks = BertClassifier(num_classes=5)
model_risks.to(device)
optimizer_risks = torch.optim.AdamW(model_risks.classifier.parameters(), lr=2e-4)
criterion_risks = nn.CrossEntropyLoss()
model = model_risks
model.to(device)
optimizer = optimizer_risks
criterion = criterion_risks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
X_text_risks = df['Решение кейса'].fillna('').values
y_risks = (df['Анализ рисков'].values - 1).astype("int64")

X_train_text_risks, X_test_text_risks, y_train_risks, y_test_risks = train_test_split(
    X_text_risks, y_risks, test_size=0.2, random_state=42
)
X_text_dataset_risks = TextDataset(X_train_text_risks, y_train_risks, tokenizer)
X_val_dataset_risks = TextDataset(X_test_text_risks, y_test_risks, tokenizer)
X_text_dataloader_risks = DataLoader(X_text_dataset_risks, batch_size=16, shuffle=True)
X_val_dataloader_risks = DataLoader(X_val_dataset_risks, batch_size=16, shuffle=False)

In [23]:
best_loss_risks = float("inf")
for epoch in range(10):
  train_loss = train(X_text_dataloader_risks)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_risks)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_risks:
    best_loss_risks = val_loss
    torch.save(model_risks.state_dict(), "best_model_risks.pt")


train_loss= 1.5557079076766969
val_loss= 1.5189199447631836
val_acc= 0.38
train_loss= 1.5257661771774291
val_loss= 1.5058429070881434
val_acc= 0.38
train_loss= 1.508184051513672
val_loss= 1.493669765336173
val_acc= 0.38
train_loss= 1.493993535041809
val_loss= 1.4833514860698156
val_acc= 0.41
train_loss= 1.482080535888672
val_loss= 1.4937912906919206
val_acc= 0.42
train_loss= 1.4645405292510987
val_loss= 1.4805117675236292
val_acc= 0.42
train_loss= 1.4460468816757202
val_loss= 1.4775787251336234
val_acc= 0.42
train_loss= 1.4327795600891113
val_loss= 1.4601940938404627
val_acc= 0.42
train_loss= 1.4182572984695434
val_loss= 1.4505895376205444
val_acc= 0.4
train_loss= 1.4027939796447755
val_loss= 1.4517686537333898
val_acc= 0.41


In [24]:
model_risks.load_state_dict(torch.load("best_model_risks.pt"))
model_risks.to(device)
model_risks.eval()
_, _, predictions = evals(X_val_dataloader_risks)
true_labels = []
for batch in X_val_dataloader_risks:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       1.00      0.18      0.31        22
           1       0.00      0.00      0.00        12
           2       0.39      0.95      0.55        38
           3       0.00      0.00      0.00        22
           4       0.00      0.00      0.00         6

    accuracy                           0.40       100
   macro avg       0.28      0.23      0.17       100
weighted avg       0.37      0.40      0.28       100

MAE 0.84


In [25]:
model_proves = BertClassifier(num_classes=5)
model_proves.to(device)
optimizer_proves = torch.optim.AdamW(model_proves.classifier.parameters(), lr=2e-4)
criterion_proves = nn.CrossEntropyLoss()
model = model_proves
model.to(device)
optimizer = optimizer_proves
criterion = criterion_proves


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
X_text_proves = df['Решение кейса'].fillna('').values
y_proves = (df['Доказательства'].values - 1).astype("int64")

X_train_text_proves, X_test_text_proves, y_train_proves, y_test_proves = train_test_split(
    X_text_proves, y_proves, test_size=0.2, random_state=42
)
X_text_dataset_proves = TextDataset(X_train_text_proves, y_train_proves, tokenizer)
X_val_dataset_proves = TextDataset(X_test_text_proves, y_test_proves, tokenizer)
X_text_dataloader_proves = DataLoader(X_text_dataset_proves, batch_size=16, shuffle=True)
X_val_dataloader_proves = DataLoader(X_val_dataset_proves, batch_size=16, shuffle=False)

In [27]:
best_loss_proves = float("inf")
for epoch in range(10):
  train_loss = train(X_text_dataloader_proves)
  print("train_loss=", train_loss)
  val_loss, val_acc, predictions = evals(X_val_dataloader_proves)
  print("val_loss=", val_loss)
  print("val_acc=", val_acc)
  if val_loss <= best_loss_proves:
    best_loss_proves = val_loss
    torch.save(model_proves.state_dict(), "best_model_proves.pt")


train_loss= 1.5791192960739135
val_loss= 1.6087919814246041
val_acc= 0.25
train_loss= 1.5411883354187013
val_loss= 1.6053266354969569
val_acc= 0.25
train_loss= 1.5237300157546998
val_loss= 1.5585596902029855
val_acc= 0.34
train_loss= 1.5101391983032226
val_loss= 1.5365464857646398
val_acc= 0.35
train_loss= 1.5080970001220704
val_loss= 1.524219581059047
val_acc= 0.35
train_loss= 1.45882266998291
val_loss= 1.5270372118268694
val_acc= 0.35
train_loss= 1.4481796503067017
val_loss= 1.501506311552865
val_acc= 0.36
train_loss= 1.4266124153137207
val_loss= 1.4828575168337141
val_acc= 0.36
train_loss= 1.4104590177536012
val_loss= 1.454303468976702
val_acc= 0.36
train_loss= 1.3851968908309937
val_loss= 1.452040774481637
val_acc= 0.36


In [28]:
model_proves.load_state_dict(torch.load("best_model_proves.pt"))
model_proves.to(device)
model_proves.eval()
_, _, predictions = evals(X_val_dataloader_proves)
true_labels = []
for batch in X_val_dataloader_proves:
  labels = batch["labels"].cpu().numpy()
  true_labels.extend(labels)
print(classification_report(true_labels, predictions))
print("MAE", mean_absolute_error(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.73      0.61      0.67        18
           1       0.00      0.00      0.00        22
           2       0.00      0.00      0.00        23
           3       0.29      1.00      0.45        25
           4       0.00      0.00      0.00        12

    accuracy                           0.36       100
   macro avg       0.21      0.32      0.22       100
weighted avg       0.21      0.36      0.23       100

MAE 0.96


In [29]:
new_results_df = pd.DataFrame(columns=['Id','Текст_решения','Анализ ЦА','Проработка решения','Финансовая модель и метрики','Анализ рынков','Доказательства','Предсказанная_оценка','Дата'])

In [ ]:
import re

In [ ]:
dec = '''МОЙ ПРОЕКТ

Я хочу сделать бота для школьников. Он будет помогать решать задачи.

Целевая аудитория - школьники. Им это нужно для учебы.

Мое решение - бот в телеграме. Он будет бесплатный. Похожих ботов нет.

Финансы: разработка стоит примерно 500 тысяч рублей. Потом будем зарабатывать на рекламе.

Риски: могут появиться конкуренты. Будем делать лучше.

Доказательства: я сам учился в школе и знаю, что это нужно. Многие мои друзья тоже так думают.'''

In [30]:
def get_prediction_audience(text):
  model_audience.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_audience, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [31]:
def get_prediction_solution(text):
  model_sol.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_sol, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [32]:
def get_prediction_finance(text):
  model_finance.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  model.load_state_dict(torch.load("best_model_audience.pt", map_location=device))
  preds = predicts([text], model_finance, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [33]:
def get_prediction_risks(text):
  model_risks.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_risks, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [34]:
def get_prediction_proves(text):
  model_proves.eval()
  if pd.isna(text) or not isinstance(text, str):
        text = ""
  preds = predicts([text], model_proves, tokenizer, device, label_encoder=None)
  score = int(preds[0])
  return score


In [35]:
df_test_cases['pred_audience'] = df_test_cases['Решение кейса'].apply(get_prediction_audience)
df_test_cases['pred_sol'] = df_test_cases['Решение кейса'].apply(get_prediction_solution)
df_test_cases['pred_finance'] = df_test_cases['Решение кейса'].apply(get_prediction_finance)
df_test_cases['pred_risks'] = df_test_cases['Решение кейса'].apply(get_prediction_risks)
df_test_cases['pred_proves'] = df_test_cases['Решение кейса'].apply(get_prediction_proves)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [36]:
#ЦА
accuracy_audience = accuracy_score(df_test_cases['ЦА'], df_test_cases['pred_audience'])
f1micro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='micro')
f1macro_audience = f1_score(df_test_cases['ЦА'], df_test_cases['pred_audience'], average='macro')
print('accuracy_audience= ', accuracy_audience)
print('f1micro_audience= ', f1micro_audience)
print('f1macro_audience= ', f1macro_audience)
print(classification_report(df_test_cases['ЦА'], df_test_cases['pred_audience']))
print("MAE=", mean_absolute_error(df_test_cases['ЦА'], df_test_cases['pred_audience']))

accuracy_audience=  0.2230769230769231
f1micro_audience=  0.2230769230769231
f1macro_audience=  0.11903511076087908
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        23
           2       1.00      0.04      0.07        26
           3       0.25      0.44      0.32        34
           4       0.22      0.59      0.32        22
           5       0.00      0.00      0.00        25

    accuracy                           0.22       130
   macro avg       0.25      0.18      0.12       130
weighted avg       0.30      0.22      0.15       130

MAE= 1.1384615384615384


In [37]:
accuracy_sol = accuracy_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'])
f1micro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='micro')
f1macro_sol = f1_score(df_test_cases['Проработка решения'], df_test_cases['pred_sol'], average='macro')
print('accuracy_sol= ', accuracy_sol)
print('f1micro_sol= ', f1micro_sol)
print('f1macro_sol= ', f1macro_sol)
print(classification_report(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))
print("MAE=", mean_absolute_error(df_test_cases['Проработка решения'], df_test_cases['pred_sol']))

accuracy_sol=  0.23846153846153847
f1micro_sol=  0.23846153846153847
f1macro_sol=  0.14874003189792664
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        21
           2       0.27      0.53      0.36        32
           3       0.28      0.23      0.25        30
           4       0.28      0.28      0.28        25
           5       0.00      0.00      0.00        22

    accuracy                           0.24       130
   macro avg       0.14      0.17      0.15       130
weighted avg       0.18      0.24      0.20       130

MAE= 1.2461538461538462


In [38]:
accuracy_finance = accuracy_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'])
f1micro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='micro')
f1macro_finance = f1_score(df_test_cases['Финансовая модель'], df_test_cases['pred_finance'], average='macro')
print('accuracy_finance= ', accuracy_finance)
print('f1micro_finance= ', f1micro_finance)
print('f1macro_finance= ', f1macro_finance)
print(classification_report(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))
print("MAE=", mean_absolute_error(df_test_cases['Финансовая модель'], df_test_cases['pred_finance']))

accuracy_finance=  0.19230769230769232
f1micro_finance=  0.19230769230769232
f1macro_finance=  0.0785279805352798
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        23
           2       0.21      0.67      0.32        33
           3       0.33      0.10      0.15        31
           4       0.00      0.00      0.00        24
           5       0.00      0.00      0.00        19

    accuracy                           0.19       130
   macro avg       0.09      0.13      0.08       130
weighted avg       0.13      0.19      0.12       130

MAE= 1.353846153846154


In [39]:
accuracy_risks = accuracy_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'])
f1micro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='micro')
f1macro_risks = f1_score(df_test_cases['Анализ рынков'], df_test_cases['pred_risks'], average='macro')
print('accuracy_risks= ', accuracy_risks)
print('f1micro_risks= ', f1micro_risks)
print('f1macro_risks= ', f1macro_risks)
print(classification_report(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))
print("MAE=", mean_absolute_error(df_test_cases['Анализ рынков'], df_test_cases['pred_risks']))

accuracy_risks=  0.2153846153846154
f1micro_risks=  0.2153846153846154
f1macro_risks=  0.07075268817204301
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.20      0.04      0.06        26
           2       0.23      0.84      0.36        32
           3       0.00      0.00      0.00        30
           4       0.00      0.00      0.00        27
           5       0.00      0.00      0.00        15

    accuracy                           0.22       130
   macro avg       0.07      0.15      0.07       130
weighted avg       0.10      0.22      0.10       130

MAE= 1.2769230769230768


In [40]:
accuracy_proves = accuracy_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'])
f1micro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='micro')
f1macro_proves = f1_score(df_test_cases['Доказательства'], df_test_cases['pred_proves'], average='macro')
print('accuracy_proves= ', accuracy_proves)
print('f1micro_proves= ', f1micro_proves)
print('f1macro_proves= ', f1macro_proves)
print(classification_report(df_test_cases['Доказательства'], df_test_cases['pred_proves']))
print("MAE", mean_absolute_error(df_test_cases['Доказательства'], df_test_cases['pred_proves']))


accuracy_proves=  0.14615384615384616
f1micro_proves=  0.14615384615384616
f1macro_proves=  0.07767722473604827
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00        18
           2       0.00      0.00      0.00        38
           3       0.17      0.40      0.24        25
           4       0.15      0.47      0.23        19
           5       0.00      0.00      0.00        30

    accuracy                           0.15       130
   macro avg       0.05      0.15      0.08       130
weighted avg       0.05      0.15      0.08       130

MAE 1.176923076923077


In [ ]:
sec_dec = """АНАЛИЗ ЦА:
Выделено 3 сегмента:
- Школьники 10-11 класс (45%)
- Студенты (35%)
- Учителя (20%)
Проведен опрос 100 человек.

РЕШЕНИЕ:
Экосистема из 2 продуктов: телеграм бот + дашборд для учителей.
Партнеры: Яндекс.Образование, МФТИ.

ФИНАНСЫ:
CAPEX: 2 млн руб. CAC = 100 руб, LTV = 2000 руб.
Окупаемость: 10 месяцев.

РИСКИ:
Технические (30%) - резервный сервер.

ДОКАЗАТЕЛЬСТВА:
Пилот в школе: NPS = 65.
"""

In [42]:
df_test_cases["Predict_score"] = round((df_test_cases['pred_audience'] + df_test_cases['pred_sol'] + df_test_cases['pred_finance'] + df_test_cases['pred_risks'] + df_test_cases['pred_proves'])/5)

In [43]:
print("MAE= ",mean_absolute_error(df_test_cases["Оценка"], df_test_cases["Predict_score"]))

MAE=  0.9538461538461539


In [ ]:
sol_test = """ Я выбрала отрасль кофеен и кофе-точек с собой (кофе на вынос, кофейные киоски, небольшие кофейни на 2–5 столиков). ЦА разделена на две группы. Первая группа — это микробизнес: кофе-байки и кофе-островки с одним сотрудником, работают как ИП или самозанятые. Их проблема в том, что кофейные зерна нужно покупать свежей обжарки каждую неделю, а хорошие обжарщики требуют предоплату 100% за партию от 5 кг, это около 20–30 тысяч рублей единовременно, что для маленькой точки с ежедневной выручкой 5–7 тысяч рублей чувствительно. Вторая группа — это малый бизнес: кофейни с посадочными местами и штатом 2–5 бариста. Их проблема в том, что сезонность спроса сильно различается: зимой продажи выше за счет горячих напитков, а летом люди покупают холодный кофе и чаще берут с собой, но в межсезонье (апрель и октябрь) падение выручки достигает 30%, при этом аренду и зарплату платить надо. Я опиралась на данные исследования «Рынок кофеен России 2024» от компании CoffeeData: рост рынка на 15% за год, количество кофеен достигло 12 тысяч, из них 70% — малый и микробизнес. Также я посмотрел открытую статистику по кофейному рынку на сайте Росстата и несколько статей в профильных телеграм-каналах. В качестве решения я предлагаю продукт «Альфа.Кофе»: кредит на закупку зёрен с отсрочкой первого платежа на 30 дней и с льготным периодом 0% на первые две недели, а также сезонный овердрафт на покрытие аренды в межсезонье на сумму до 100 тысяч рублей. Партнеры — два крупных обжарщика зерна «Coffe Lab» и «Torrefacto», с которыми можно договориться о более выгодных ценах для клиентов банка. Отличие от конкурентов в том, что ни у Сбера, ни у Т-Банка нет специального продукта для кофеен с отсрочкой именно под зерно. По финансам: разработка кредитного продукта обойдется примерно в 6 миллионов рублей, интеграция с партнерами-обжарщиками — в 2 миллиона, маркетинг в кофейных чатах и через конференции бариста — в 2 миллиона. Прогноз: 350 клиентов в первый год, средний доход с клиента — 20 тысяч рублей (за счет процентов по кредиту и эквайринга), выручка — 7 миллионов рублей. Окупаемость — примерно 2,5 года. Риски: конкурентный — другие банки могут запустить похожие продукты; кредитный — часть клиентов может не вернуть деньги, но мы будем проверять по выписке с кофемашины, кто сколько продает. В доказательство я использовал данные CoffeeData и результаты пары интервью с владельцами кофеен в своем городе."""

In [ ]:
predicted_score_audience = get_prediction_audience(sec_dec)
predicted_score_sol = get_prediction_solution(sec_dec)
predicted_score_finance = get_prediction_finance(sec_dec)
predicted_score_risks = get_prediction_risks(sec_dec)
predicted_score_proves = get_prediction_proves(sec_dec)
predicted_score_audience2 = get_prediction_audience(sec_dec)
predicted_score_sol2 = get_prediction_solution(sec_dec)
predicted_score_finance2 = get_prediction_finance(sec_dec)
predicted_score_risks2 = get_prediction_risks(sec_dec)
predicted_score_proves2 = get_prediction_proves(sec_dec)
new_row = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience,
    'Проработка решения': predicted_score_sol,
    'Финансовая модель и метрики': predicted_score_finance,
    'Анализ рынков': predicted_score_risks,
    'Доказательства': predicted_score_proves,
    'Предсказанная_оценка': round((predicted_score_audience + predicted_score_sol +
                      predicted_score_finance + predicted_score_risks +
                      predicted_score_proves) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_row2 = pd.DataFrame([{
    'Id': 1,
    'Текст_решения': sec_dec,
    'Анализ ЦА': predicted_score_audience2,
    'Проработка решения': predicted_score_sol2,
    'Финансовая модель и метрики': predicted_score_finance2,
    'Анализ рынков': predicted_score_risks2,
    'Доказательства': predicted_score_proves2,
    'Предсказанная_оценка': round((predicted_score_audience2 + predicted_score_sol2 +
                      predicted_score_finance2 + predicted_score_risks2 +
                      predicted_score_proves2) / 5),
    'Дата': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}])
new_results_df = pd.concat([new_results_df, new_row, new_row2], ignore_index=True)
new_results_df.to_excel("новая_таблица.xlsx", index=False)


/tmp/ipykernel_3876/1005490930.py:8: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)
/tmp/ipykernel_3876/1097642392.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(score)


In [ ]:
new_results_df

,Id,Текст_решения,Анализ ЦА,Проработка решения,Финансовая модель и метрики,Анализ рынков,Доказательства,Предсказанная_оценка,Дата
0,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
1,1,АНАЛИЗ ЦА:\nВыделено 3 сегмента:\n- Школьники ...,5,4,4,4,4,4,2026-06-22 15:34:44
